# SCADA ingest from OneLake tables

Lee la tabla SCADA limpia de cada planta (vía shortcut OneLake) y vuelca a
`proc_scada` aplicando el mapeo declarado en `plants_config.xlsx → 02_Sensors_Mapping`.

**Parámetros** (los puede sobreescribir el pipeline):
- `plant_filter` (`'PL2'` → sólo esa planta; `None` → todas las que tengan
  `pr_execution = TRUE` en 01_Plants y `scada_source_type = 'onelake_table'`)
- `period_start` / `period_end` (`'YYYY-MM-DD HH:MM:SS'` o `None` → todo el rango disponible)


In [ ]:
# Parámetros
plant_filter  = None
period_start  = None
period_end    = None


In [ ]:
import io
import pandas as pd
from datetime import datetime
from pathlib import Path
from delta.tables import DeltaTable
from pyspark.sql import Row
from pyspark.sql.functions import col, lit, current_timestamp

LAKEHOUSE_FILES = '/lakehouse/default/Files'
CONFIG_XLSX     = f'{LAKEHOUSE_FILES}/Raw PVSyst1/CONFIG/plants_config.xlsx'


ABFS_BASE = "abfss://0c4c3bbe-a6ae-404f-b1b7-7052bacfea11@onelake.dfs.fabric.microsoft.com/ae7e1482-8f15-4b1b-8954-8ee500382179"


def _read(sheet):
    """Lee una hoja del Excel de configuración via Spark con ruta ABFS completa."""
    local = '/tmp/plants_config.xlsx'
    if not Path(local).exists():
        abfs_path = ABFS_BASE + '/Files/Raw PVSyst1/CONFIG/plants_config.xlsx'
        row = spark.read.format('binaryFile').load(abfs_path).first()
        with open(local, 'wb') as _f:
            _f.write(row['content'])
    return pd.read_excel(local, sheet_name=sheet)

def get_active_plants():
    df = _read('01_Plants')
    df = df[df['pr_execution'].astype(str).str.upper().isin(['TRUE','VERDADERO','1','SÍ','SI'])]
    if plant_filter:
        df = df[df['plant_code'] == plant_filter]
    return df.to_dict(orient='records')


def get_scada_mapping(plant_code):
    m = _read('02_Sensors_Mapping')
    rows = m[m['plant_code'] == plant_code]
    if rows.empty:
        rows = m[m['plant_code'] == 'COMMON']
    mapping = {}
    for _, r in rows.iterrows():
        orig = str(r['columna_original']).strip() if pd.notna(r['columna_original']) else None
        dest = str(r['columna_destino']).strip()  if pd.notna(r['columna_destino'])  else None
        if orig and dest:
            mapping[orig] = dest
    return mapping

## 1 · Leer la tabla OneLake por planta y aplicar el mapeo

In [ ]:
def read_and_normalize_scada(table_name, mapping, period_start=None, period_end=None):
    sdf = spark.table(table_name)

    for orig, dest in mapping.items():
        if orig in sdf.columns and orig != dest:
            sdf = sdf.withColumnRenamed(orig, dest)

    if 'Date' not in sdf.columns:
        raise ValueError(f"La tabla {table_name} no contiene columna Date tras el mapeo")

    if period_start:
        sdf = sdf.filter(col('Date') >= lit(period_start).cast('timestamp'))
    if period_end:
        sdf = sdf.filter(col('Date') <= lit(period_end).cast('timestamp'))

    # Quedarse sólo con las columnas definidas en el Excel (columna_destino) + Date
    keep = list(mapping.values())
    if 'Date' not in keep:
        keep = ['Date'] + keep
    sdf = sdf.select(*[c for c in keep if c in sdf.columns])
    return sdf

## 2 · MERGE en proc_scada por planta

In [ ]:
def merge_into_silver(sdf, plant_code, source_table):
    n = sdf.count()  # capturamos antes de añadir columnas de control
    sdf = (
        sdf
        .withColumn('plant_code',  lit(plant_code))
        .withColumn('source_file', lit(source_table))
        .withColumn('ingested_at', current_timestamp())
    )
    target = DeltaTable.forName(spark, 'proc_scada')
    cond = "t.plant_code = s.plant_code AND t.Date = s.Date"
    (target.alias('t')
        .merge(sdf.alias('s'), cond)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    return n


results = []
for p in get_active_plants():
    plant   = p['plant_code']
    table   = 'gold.' + str(p['scada_table_name']).strip()
    print(f"\n→ {plant}: leyendo tabla {table}")
    try:
        mapping = get_scada_mapping(plant)
        sdf     = read_and_normalize_scada(table, mapping, period_start, period_end)
        n       = merge_into_silver(sdf, plant, table)
        status, error = 'ok', None
        print(f"  {n:,} filas merged en proc_scada")
    except Exception as e:
        n = 0; status = 'failed'; error = str(e)[:500]
        print(f"  FAILED: {error}")
    results.append(Row(
        plant_code=plant, file_path=f'onelake://{table}',
        file_kind='scada', file_size=None, row_count=n,
        period_start=None, period_end=None, resolution=None,
        status=status, error=error, ingested_at=datetime.utcnow(),
    ))

if results:
    rows_pd = pd.DataFrame([r.asDict() for r in results])
    for tcol in ('period_start', 'period_end', 'ingested_at'):
        rows_pd[tcol] = pd.to_datetime(rows_pd[tcol], errors='coerce')
    spark.createDataFrame(rows_pd).write.mode('append').format('delta').saveAsTable('ingested_files')
    print(f"\nLogged {len(results)} filas en ingested_files")

## 3 · Salida del notebook

In [ ]:
import json, notebookutils
summary = {
    'plants_processed': len(results),
    'ok':     sum(1 for r in results if r['status'] == 'ok'),
    'failed': sum(1 for r in results if r['status'] == 'failed'),
}
print(json.dumps(summary, indent=2))
notebookutils.notebook.exit(json.dumps(summary))
